In [2]:
# %% [markdown]
# ## Importing Relevant Libraries
# Organizing our imports into logical groups for better code organization and readability

# %% [code]
# Standard library imports
import os
import getpass
from datetime import datetime, timedelta
from operator import itemgetter
from uuid import uuid4

# Document loading and processing
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector stores and embeddings
from langchain_community.vectorstores import Qdrant
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models
from langchain_openai import OpenAIEmbeddings

# Retrieval components
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import EnsembleRetriever, ParentDocumentRetriever
from langchain_cohere import CohereRerank
from langchain.storage import InMemoryStore

# LLM and chat components
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Evaluation tools
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import RunConfig, EvaluationDataset, evaluate
from ragas.testset import TestsetGenerator
from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    NoiseSensitivity
)




***Get all the key info***

In [3]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [4]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

In [5]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

##  Data Collection and Preparation

Using the PDF !

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today

In [6]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` 

In [7]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

In [8]:
### Loading the pdf data set
path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

In [9]:
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [10]:
import os
import pandas as pd

# Check if testset already exists in golden_dataset directory - this prevents wasteful cycles in regeneration 
testset_path = "golden_dataset/test_set.csv"

if os.path.exists(testset_path):
    print("Loading existing testset from golden_dataset/test_set.csv")
    test_set = pd.read_csv(testset_path)
    print(f"Loaded {len(test_set)} questions from existing testset")
else:
    print("No existing testset found. Generating new testset...")
    generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
    dataset = generator.generate_with_langchain_docs(docs, testset_size=10)
    test_set = dataset.to_pandas()
    print(f"Generated new testset with {len(test_set)} questions")

Loading existing testset from golden_dataset/test_set.csv
Loaded 12 questions from existing testset


In [11]:
print(test_set)

                                           user_input  \
0   What role does Pew Research Center play in und...   
1   How is ChatGPT primarily utilized in the workp...   
2   What does Appendix D report regarding variatio...   
3   What does the term 'Seeking Information' refer...   
4   How do the changing usage patterns between Jun...   
5   How did the rapid adoption of ChatGPT since it...   
6   How do the trends in monthly message statistic...   
7   How does ChatGPT usage vary across different o...   
8   how november 2022 launch of ChatGPT and novemb...   
9   wHEN in NOvember 2022 was ChatGPT lauched and ...   
10  Based on the findings of Handa et al. (2025), ...   
11  Considering the rapid growth of ChatGPT usage ...   

                                   reference_contexts  \
0   ['Introduction ChatGPT launched in November 20...   
1   ['Table 1: ChatGPT daily message counts (milli...   
2   ['Variation by Occupation Figure 23 presents v...   
3   ['Conclusion This paper st

In [12]:
#save the test set into a file so that we don't lose it if Ragas decides to go on strike again
import pandas as pd
if not os.path.exists("golden_dataset/test_set.csv"):
    test_set.to_csv("golden_dataset/test_set.csv", index=False)
    print("Testset saved to golden_dataset/test_set.csv")
else:
    print("Testset file already exists, skipping save")


Testset file already exists, skipping save


In [13]:
# Import the text splitter that recursively splits text into smaller chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Create a text splitter that will:
# - Split documents into chunks of 500 characters
# - Have an overlap of 50 characters between chunks to maintain context
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

# Split our documents into smaller chunks for better processing
# This helps with:
# - More focused retrieval
# - Staying within token limits
# - More precise context matching
rag_documents = text_splitter.split_documents(docs)

In [14]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [16]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

In [ ]:
#using knowledge graphs since Ragas wasnt working with the abstracted version
'''
from ragas.testset.graph import KnowledgeGraph
from ragas.testset.graph import Node, NodeType
from ragas.testset.transforms import default_transforms, apply_transforms
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

kg = KnowledgeGraph()
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm),1),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0),
]
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()
'''

In [15]:
# Create Qdrant vector store with non-semantic chunks
vectorstore_pdf = Qdrant.from_documents(
    rag_documents,
    embeddings,
    location=":memory:",
    collection_name="non-semantic_chunks"
)

print(f"non-semantic vector store created: {len(rag_documents)} documents")

non-semantic vector store created: 275 documents


In [17]:
#Update all the retrievers to use the new vector store
bm25_retriever_pdf = BM25Retriever.from_documents(docs)
bm25_retrieval_pdf_chain = (
    {"context": itemgetter("question") | bm25_retriever_pdf, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)


In [18]:
naive_retriever_pdf = vectorstore_pdf.as_retriever(search_kwargs={"k" : 10})
naive_retrieval_pdf_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever_pdf, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [19]:
compressor = CohereRerank(model="rerank-v3.5", top_n=7)
compression_retriever_pdf = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever_pdf
)
contextual_compression_retrieval_pdf_chain = (
    {"context": itemgetter("question") | compression_retriever_pdf, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
multi_query_retriever_pdf = MultiQueryRetriever.from_llm(
    retriever=naive_retriever_pdf, llm=chat_model
)
multi_query_retrieval_pdf_chain = (
    {"context": itemgetter("question") | multi_query_retriever_pdf, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
parent_docs = docs

client_pdf = QdrantClient(location=":memory:")
client_pdf.create_collection(
    collection_name="pdf_docs",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore_pdf = QdrantVectorStore(
    collection_name="pdf_docs", 
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"), 
    client=client_pdf
)

# Create parent document retriever
parent_document_retriever_pdf = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore_pdf,
    docstore=InMemoryStore(),
    child_splitter=child_splitter, 
)

# Add the combined documents
parent_document_retriever_pdf.add_documents(docs, ids=None)

parent_document_retrieval_pdf_chain = (
    {"context": itemgetter("question") | parent_document_retriever_pdf, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)


In [22]:
retriever_pdf_list = [bm25_retriever_pdf, naive_retriever_pdf, parent_document_retriever_pdf, compression_retriever_pdf, multi_query_retriever_pdf]
equal_weighting = [1/len(retriever_pdf_list)] * len(retriever_pdf_list)

ensemble_retriever_pdf = EnsembleRetriever(
    retrievers=retriever_pdf_list, weights=equal_weighting
)

ensemble_retrieval_pdf_chain = (
    {"context": itemgetter("question") | ensemble_retriever_pdf, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [23]:
test_set.head()

,user_input,reference_contexts,reference,synthesizer_name
0,What role does Pew Research Center play in und...,['Introduction ChatGPT launched in November 20...,The provided context mentions Pew Research Cen...,single_hop_specifc_query_synthesizer
1,How is ChatGPT primarily utilized in the workp...,['Table 1: ChatGPT daily message counts (milli...,"ChatGPT is most commonly used for Writing, whi...",single_hop_specifc_query_synthesizer
2,What does Appendix D report regarding variatio...,['Variation by Occupation Figure 23 presents v...,Appendix D presents variation in ChatGPT usage...,single_hop_specifc_query_synthesizer
3,What does the term 'Seeking Information' refer...,['Conclusion This paper studies the rapid grow...,"In the context of ChatGPT usage, 'Seeking Info...",single_hop_specifc_query_synthesizer
4,How do the changing usage patterns between Jun...,['<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (...,The data shows that from June 2024 to June 202...,multi_hop_abstract_query_synthesizer


In [24]:
# Define LLM and Run Config
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
custom_run_config = RunConfig(timeout=720)

print("LLM and Run Config configured:")
print(f"Model: gpt-4.1-mini")
print(f"Timeout: 720 seconds")

LLM and Run Config configured:
Model: gpt-4.1-mini
Timeout: 720 seconds


Evaluations

In [27]:
# RaGAS metric extraction with proper key mapping
DEBUG = True

def extract_ragas_metrics_final(ragas_result, ragas_metrics):
    """Correctly extract metrics from Ragas result with proper key mapping"""
    result_metrics = {}
    
    # Mapping from class names to actual keys in Ragas scores
    metric_key_mapping = {
        'LLMContextRecall': 'context_recall',
        'Faithfulness': 'faithfulness',
        'FactualCorrectness': 'factual_correctness',
        'ResponseRelevancy': 'answer_relevancy',
        'ContextEntityRecall': 'context_entity_recall',
        'NoiseSensitivity': 'noise_sensitivity_relevant'
    }
    
    if DEBUG:
        print(f"🔍 DEBUG: Extracting metrics from Ragas result...")
        print(f"🔍 DEBUG: Scores type: {type(ragas_result.scores)}")
    
    if not ragas_result.scores:
        if DEBUG:
            print("🔍 DEBUG: No scores found!")
        return {metric.__class__.__name__: None for metric in ragas_metrics}
    
    # Get the first sample's scores
    sample_scores = ragas_result.scores[0]
    if DEBUG:
        print(f"🔍 DEBUG: Sample scores keys: {list(sample_scores.keys())}")
    
    for metric in ragas_metrics:
        metric_name = metric.__class__.__name__
        if DEBUG:
            print(f"🔍 DEBUG: Looking for metric: {metric_name}")
        
        # Get the correct key for this metric
        correct_key = metric_key_mapping.get(metric_name, metric_name.lower())
        if DEBUG:
            print(f"🔍 DEBUG: Looking for key: '{correct_key}'")
        
        if correct_key in sample_scores:
            score = sample_scores[correct_key]
            # Handle NaN values
            if str(score).lower() == 'nan' or (hasattr(score, '__iter__') and str(score) == 'nan'):
                result_metrics[metric_name] = None
                if DEBUG:
                    print(f"🔍 DEBUG: Found {metric_name} = {score} (converted to None due to NaN)")
            else:
                result_metrics[metric_name] = float(score) if score is not None else None
                if DEBUG:
                    print(f"🔍 DEBUG: Found {metric_name} = {score}")
        else:
            result_metrics[metric_name] = None
            if DEBUG:
                print(f"🔍 DEBUG: {metric_name} not found (key '{correct_key}' not in scores)")
    
    return result_metrics


In [25]:
#  LangSmith project setup function
def setup_langsmith_project(name):
    """Setup LangSmith project with proper creation and error handling"""
    if os.environ.get("LANGCHAIN_TRACING_V2") != "true":
        return None
        
    retriever_project = f"{name}_retriever_eval"
    client = Client()
    
    try:
        # Delete existing project if it exists
        try:
            client.delete_project(project_name=retriever_project)
            print(f"🗑️  Deleted existing project: {retriever_project}")
            time.sleep(2)
        except:
            pass
            
        # Create new project
        project = client.create_project(project_name=retriever_project)
        print(f"✅ Created new project: {retriever_project}")
        
    except Exception as e:
        # Try to use existing project if creation fails
        try:
            project = client.read_project(project_name=retriever_project)
            print(f"ℹ️  Using existing project: {retriever_project}")
        except:
            print(f"❌ Could not setup project: {retriever_project}")
            return None
            
    os.environ["LANGCHAIN_PROJECT"] = retriever_project
    print(f"📁 LangSmith Project: {retriever_project}\n")
    return retriever_project

print("LangSmith project setup function defined!")


LangSmith project setup function defined!


In [ ]:
# Returns empty metrics dictionary with zero values for when metrics are unavailable
def unavailable_metrics():
    """Return empty metrics dictionary with zero values"""
    return {
        'p50_latency_seconds': 0.0,
        'p99_latency_seconds': 0.0,
        'cost_per_query_usd': 0.0,
        'prompt_cost_per_query_usd': 0.0,
        'completion_cost_per_query_usd': 0.0,
        'total_tokens_per_query': 0.0,
    }

# Extracts cost and latency metrics from LangSmith runs, with fallback options if metrics are unavailable
# Handles errors gracefully and returns default metrics if anything fails
def get_langsmith_metrics(project_name: str, num_queries: int) -> dict:
    """Extract cost and latency metrics from LangSmith with fallback options"""
    try:
        if not project_name:
            return unavailable_metrics()
            
        client = Client()
        
        # Get project and runs
        try:
            project = client.read_project(project_name=project_name, include_stats=True)
            runs = list(client.list_runs(project_name=project_name, limit=100))
            if not runs:
                return unavailable_metrics()
        except Exception as e:
            print(f"Error accessing project {project_name}: {e}")
            return unavailable_metrics()

        # Calculate costs and tokens from runs
        total_cost = prompt_cost = completion_cost = total_tokens = 0
        
        # Try direct attribute access first
        for run in runs:
            total_cost += getattr(run, 'total_cost', 0) or 0
            prompt_cost += getattr(run, 'prompt_cost', 0) or 0 
            completion_cost += getattr(run, 'completion_cost', 0) or 0
            total_tokens += getattr(run, 'total_tokens', 0) or 0

        # If no data, try extracting from run.extra
        if total_cost == 0 and total_tokens == 0:
            for run in runs:
                if run.extra:
                    if 'cost' in run.extra:
                        cost_data = run.extra['cost']
                        total_cost += cost_data.get('total_cost', 0)
                        prompt_cost += cost_data.get('prompt_cost', 0)
                        completion_cost += cost_data.get('completion_cost', 0)
                    if 'usage' in run.extra:
                        total_tokens += run.extra['usage'].get('total_tokens', 0)

        # Get latencies
        latency_p50 = getattr(project, 'latency_p50', None)
        latency_p99 = getattr(project, 'latency_p99', None)

        # Calculate latencies from runs if not in project stats
        if not latency_p50 or not latency_p99:
            latencies = [run.latency for run in runs if run.latency is not None]
            if latencies:
                latencies.sort()
                p50_idx = int(len(latencies) * 0.5)
                p99_idx = min(int(len(latencies) * 0.99), len(latencies) - 1)
                latency_p50 = timedelta(seconds=latencies[p50_idx])
                latency_p99 = timedelta(seconds=latencies[p99_idx])

        return {
            'p50_latency_seconds': latency_p50.total_seconds() if latency_p50 else 0.0,
            'p99_latency_seconds': latency_p99.total_seconds() if latency_p99 else 0.0,
            'cost_per_query_usd': total_cost / num_queries if num_queries > 0 else 0.0,
            'prompt_cost_per_query_usd': prompt_cost / num_queries if num_queries > 0 else 0.0,
            'completion_cost_per_query_usd': completion_cost / num_queries if num_queries > 0 else 0.0,
            'total_tokens_per_query': total_tokens / num_queries if num_queries > 0 else 0.0,
        }

    except Exception as e:
        print(f"Error extracting metrics: {e}")
        return unavailable_metrics()


In [ ]:
# Import required modules
import time
from langsmith import Client
from langchain_core.tracers.context import tracing_v2_enabled

# Function to evaluate a retrieval method using both Ragas and LangSmith metrics
# Takes a method name, chain, test set, LLM evaluator, and run config
# Returns a dictionary of metrics including latency, cost, and various Ragas scores
# Handles rate limiting for specific retrievers and includes error handling
def evaluate_retriever_method(method_name, chain, test_set, evaluator_llm, run_config):
    """Corrected evaluation function with fixed LangSmith project setup"""
    
    print(f"\nEvaluating {method_name}...")
    start_time = time.time()
    
    # Setup LangSmith project for this retriever using the fixed function
    project_name = setup_langsmith_project(method_name.replace(' ', '_').lower())
    
    # Initialize result dictionary with Ragas metrics
    result_metrics = {
        'Method': method_name, 'Status': 'Failed', 'Evaluation_Time_seconds': 0,
        'Total_Cost_USD': 0, 'Avg_Latency_ms': 0, 'Total_Tokens': 0, 'Cost_per_Question': 0,
        'LLMContextRecall': None, 'Faithfulness': None, 'FactualCorrectness': None,
        'ResponseRelevancy': None, 'ContextEntityRecall': None, 'NoiseSensitivity': None
    }
    
    try:
        # Generate responses
        responses = []
        contexts = []
        
        for idx, row in test_set.iterrows():
            with tracing_v2_enabled(project_name=project_name): # Fixed: Added missing colon and block
                # Apply rate limit delay for Cohere and Ensemble retrievers before each question
                if method_name.lower() in ['contextual compression', 'ensemble retrieval']:
                    print(f"⏳ Applying 30-second rate limit delay for {method_name} before question {idx+1}...")
                    time.sleep(30)
                    print(f"✅ Rate limit delay completed for {method_name} - question {idx+1}")
                
                result = chain.invoke({"question": row['user_input']})
                response = result["response"].content
                context = [str(doc.page_content) for doc in result["context"]]
                responses.append(response)
                contexts.append(context)
        
        # Create evaluation dataset
        eval_df = pd.DataFrame({
            'user_input': test_set['user_input'].tolist(),
            'response': responses,
            'retrieved_contexts': contexts,
            'reference': test_set['reference'].tolist()
        })
        eval_dataset = EvaluationDataset.from_pandas(eval_df)
        
        # Define Ragas metrics
        ragas_metrics = [
            LLMContextRecall(), Faithfulness(), FactualCorrectness(),
            ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()
        ]
        
        # Run Ragas evaluation
        try:
            ragas_result = evaluate(
                dataset=eval_dataset, metrics=ragas_metrics, llm=evaluator_llm, run_config=run_config
            )
            
            # Extract Ragas scores using the working function
            ragas_scores = extract_ragas_metrics_final(ragas_result, ragas_metrics)
            result_metrics.update(ragas_scores)
            
        except Exception as eval_error:
            print(f"Ragas evaluation failed: {eval_error}")
            # Fallback to smaller subset
            small_eval_df = eval_df.head(5)
            small_eval_dataset = EvaluationDataset.from_pandas(small_eval_df)
            ragas_result = evaluate(
                dataset=small_eval_dataset, metrics=ragas_metrics, llm=evaluator_llm, run_config=run_config
            )
            ragas_scores = extract_ragas_metrics_final(ragas_result, ragas_metrics)
            result_metrics.update(ragas_scores)
        
        # Get LangSmith metrics using the working function
        if project_name:
            time.sleep(5)
            langsmith_metrics = get_langsmith_metrics(project_name, len(test_set))
            result_metrics.update({
                'Total_Cost_USD': langsmith_metrics['cost_per_query_usd'] * len(test_set),
                'Avg_Latency_ms': langsmith_metrics['p50_latency_seconds'] * 1000,
                'Total_Tokens': langsmith_metrics['total_tokens_per_query'] * len(test_set),
                'Cost_per_Question': langsmith_metrics['cost_per_query_usd'],
                'P50_Latency_seconds': langsmith_metrics['p50_latency_seconds'],
                'P99_Latency_seconds': langsmith_metrics['p99_latency_seconds'],
                'Prompt_Cost_per_Query': langsmith_metrics['prompt_cost_per_query_usd'],
                'Completion_Cost_per_Query': langsmith_metrics['completion_cost_per_query_usd']
            })
        else:
            print(f"⚠️ No LangSmith project available for {method_name}")
        
        elapsed_time = time.time() - start_time
        result_metrics.update({'Status': 'Success', 'Evaluation_Time_seconds': elapsed_time})
        print(f"✓ {method_name} completed in {elapsed_time:.2f} seconds")
        
    except Exception as e:
        print(f"✗ Error evaluating {method_name}: {str(e)}")
        result_metrics['Status'] = 'Failed'
        result_metrics['Evaluation_Time_seconds'] = time.time() - start_time
    
    return result_metrics


In [39]:
# QUICK TEST - Test the corrected function with one retriever


print("🔧 TESTING: Corrected evaluation function with one retriever...")
print("=" * 70)

# Test with just one retriever to verify it works
test_result = evaluate_retriever_method(
    "Naive Retrieval", 
    naive_retrieval_pdf_chain, 
    test_set.head(2),  # Use only 2 questions for testing
    evaluator_llm, 
    custom_run_config
)

print("\n🔧 TEST RESULT:")
print("=" * 50)
for key, value in test_result.items():
    print(f"{key}: {value}")


🔧 TESTING: Corrected evaluation function with one retriever...

Evaluating Naive Retrieval...
🗑️  Deleted existing project: naive_retrieval_retriever_eval
✅ Created new project: naive_retrieval_retriever_eval
📁 LangSmith Project: naive_retrieval_retriever_eval



Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

Exception raised in Job[5]: AttributeError('StringIO' object has no attribute 'statements')


🔍 DEBUG: Extracting metrics from Ragas result...
🔍 DEBUG: Scores type: <class 'list'>
🔍 DEBUG: Sample scores keys: ['context_recall', 'faithfulness', 'factual_correctness', 'answer_relevancy', 'context_entity_recall', 'noise_sensitivity_relevant']
🔍 DEBUG: Looking for metric: LLMContextRecall
🔍 DEBUG: Looking for key: 'context_recall'
🔍 DEBUG: Found LLMContextRecall = 1.0
🔍 DEBUG: Looking for metric: Faithfulness
🔍 DEBUG: Looking for key: 'faithfulness'
🔍 DEBUG: Found Faithfulness = 0.8888888888888888
🔍 DEBUG: Looking for metric: FactualCorrectness
🔍 DEBUG: Looking for key: 'factual_correctness'
🔍 DEBUG: Found FactualCorrectness = 0.84
🔍 DEBUG: Looking for metric: ResponseRelevancy
🔍 DEBUG: Looking for key: 'answer_relevancy'
🔍 DEBUG: Found ResponseRelevancy = 0.9999999999999997
🔍 DEBUG: Looking for metric: ContextEntityRecall
🔍 DEBUG: Looking for key: 'context_entity_recall'
🔍 DEBUG: Found ContextEntityRecall = 0.7499999981250001
🔍 DEBUG: Looking for metric: NoiseSensitivity
🔍 DEBUG: 

In [ ]:
# FULL EVALUATION --All retrievers
print("🚀 STARTING FULL EVALUATION WITH CORRECTED FUNCTION...")
print("=" * 80)
print("This will evaluate all retrieval methods with working Ragas and LangSmith metrics")
print("⏰ Estimated time: 15-20 minutes (including rate limits)")
print("=" * 80)

# Initialize results list
all_results = []

# Evaluate Naive Retrieval
print("\n" + "="*60)
print("1/6: EVALUATING NAIVE RETRIEVAL")
print("="*60)
result1 = evaluate_retriever_method("Naive Retrieval", naive_retrieval_pdf_chain, test_set, evaluator_llm, custom_run_config)
all_results.append(result1)

# Evaluate BM25 Retrieval
print("\n" + "="*60)
print("2/6: EVALUATING BM25 RETRIEVAL")
print("="*60)
result2 = evaluate_retriever_method("BM25 Retrieval", bm25_retrieval_pdf_chain, test_set, evaluator_llm, custom_run_config)
all_results.append(result2)



🚀 STARTING FULL EVALUATION WITH CORRECTED FUNCTION...
This will evaluate all retrieval methods with working Ragas and LangSmith metrics
⏰ Estimated time: 15-20 minutes (including rate limits)

1/6: EVALUATING NAIVE RETRIEVAL

Evaluating Naive Retrieval...
🗑️  Deleted existing project: naive_retrieval_retriever_eval
✅ Created new project: naive_retrieval_retriever_eval
📁 LangSmith Project: naive_retrieval_retriever_eval



Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

🔍 DEBUG: Extracting metrics from Ragas result...
🔍 DEBUG: Scores type: <class 'list'>
🔍 DEBUG: Sample scores keys: ['context_recall', 'faithfulness', 'factual_correctness', 'answer_relevancy', 'context_entity_recall', 'noise_sensitivity_relevant']
🔍 DEBUG: Looking for metric: LLMContextRecall
🔍 DEBUG: Looking for key: 'context_recall'
🔍 DEBUG: Found LLMContextRecall = 1.0
🔍 DEBUG: Looking for metric: Faithfulness
🔍 DEBUG: Looking for key: 'faithfulness'
🔍 DEBUG: Found Faithfulness = 0.8181818181818182
🔍 DEBUG: Looking for metric: FactualCorrectness
🔍 DEBUG: Looking for key: 'factual_correctness'
🔍 DEBUG: Found FactualCorrectness = 0.75
🔍 DEBUG: Looking for metric: ResponseRelevancy
🔍 DEBUG: Looking for key: 'answer_relevancy'
🔍 DEBUG: Found ResponseRelevancy = 0.9808066871287683
🔍 DEBUG: Looking for metric: ContextEntityRecall
🔍 DEBUG: Looking for key: 'context_entity_recall'
🔍 DEBUG: Found ContextEntityRecall = 0.7499999981250001
🔍 DEBUG: Looking for metric: NoiseSensitivity
🔍 DEBUG: 

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[1]: AttributeError('StringIO' object has no attribute 'statements')


🔍 DEBUG: Extracting metrics from Ragas result...
🔍 DEBUG: Scores type: <class 'list'>
🔍 DEBUG: Sample scores keys: ['context_recall', 'faithfulness', 'factual_correctness', 'answer_relevancy', 'context_entity_recall', 'noise_sensitivity_relevant']
🔍 DEBUG: Looking for metric: LLMContextRecall
🔍 DEBUG: Looking for key: 'context_recall'
🔍 DEBUG: Found LLMContextRecall = 1.0
🔍 DEBUG: Looking for metric: Faithfulness
🔍 DEBUG: Looking for key: 'faithfulness'
🔍 DEBUG: Found Faithfulness = nan (converted to None due to NaN)
🔍 DEBUG: Looking for metric: FactualCorrectness
🔍 DEBUG: Looking for key: 'factual_correctness'
🔍 DEBUG: Found FactualCorrectness = 0.53
🔍 DEBUG: Looking for metric: ResponseRelevancy
🔍 DEBUG: Looking for key: 'answer_relevancy'
🔍 DEBUG: Found ResponseRelevancy = 0.9979412405590059
🔍 DEBUG: Looking for metric: ContextEntityRecall
🔍 DEBUG: Looking for key: 'context_entity_recall'
🔍 DEBUG: Found ContextEntityRecall = 0.49999999875
🔍 DEBUG: Looking for metric: NoiseSensitivit

In [42]:
print(all_results)

[{'Method': 'Naive Retrieval', 'Status': 'Success', 'Evaluation_Time_seconds': 528.1506361961365, 'Total_Cost_USD': Decimal('0.2208592000000000000000000000'), 'Avg_Latency_ms': 2683.0, 'Total_Tokens': 227242.0, 'Cost_per_Question': Decimal('0.01840493333333333333333333333'), 'LLMContextRecall': 1.0, 'Faithfulness': 0.8181818181818182, 'FactualCorrectness': 0.75, 'ResponseRelevancy': 0.9808066871287683, 'ContextEntityRecall': 0.7499999981250001, 'NoiseSensitivity': 0.18181818181818182, 'P50_Latency_seconds': 2.683, 'P99_Latency_seconds': 416.11956, 'Prompt_Cost_per_Query': Decimal('0.003964666666666666666666666667'), 'Completion_Cost_per_Query': Decimal('0.01444026666666666666666666667')}, {'Method': 'BM25 Retrieval', 'Status': 'Success', 'Evaluation_Time_seconds': 312.60010170936584, 'Total_Cost_USD': Decimal('0.013953300'), 'Avg_Latency_ms': 3054.5, 'Total_Tokens': 114444.0, 'Cost_per_Question': Decimal('0.001162775'), 'LLMContextRecall': 1.0, 'Faithfulness': None, 'FactualCorrectness

In [43]:
# Evaluate Contextual Compression Retrieval (with rate limit handling)
print("\n" + "="*60)
print("3/6: EVALUATING CONTEXTUAL COMPRESSION (with rate limits)")
print("="*60)
result3 = evaluate_retriever_method("Contextual Compression", contextual_compression_retrieval_pdf_chain, test_set, evaluator_llm, custom_run_config)
all_results.append(result3)

# Evaluate Multi-Query Retrieval
print("\n" + "="*60)
print("4/6: EVALUATING MULTI-QUERY RETRIEVAL")
print("="*60)
result4 = evaluate_retriever_method("Multi-Query Retrieval", multi_query_retrieval_pdf_chain, test_set, evaluator_llm, custom_run_config)
all_results.append(result4)

# Evaluate Parent Document Retrieval
print("\n" + "="*60)
print("5/6: EVALUATING PARENT DOCUMENT RETRIEVAL")
print("="*60)
result5 = evaluate_retriever_method("Parent Document Retrieval", parent_document_retrieval_pdf_chain, test_set, evaluator_llm, custom_run_config)
all_results.append(result5)

# Evaluate Ensemble Retrieval (with rate limit handling)
print("\n" + "="*60)
print("6/6: EVALUATING ENSEMBLE RETRIEVAL (with rate limits)")
print("="*60)
result6 = evaluate_retriever_method("Ensemble Retrieval", ensemble_retrieval_pdf_chain, test_set, evaluator_llm, custom_run_config)
all_results.append(result6)

print("\n" + "=" * 80)
print("🎉 ALL EVALUATIONS COMPLETED WITH CORRECTED FUNCTION!")
print("=" * 80)



3/6: EVALUATING CONTEXTUAL COMPRESSION (with rate limits)

Evaluating Contextual Compression...
🗑️  Deleted existing project: contextual_compression_retriever_eval
✅ Created new project: contextual_compression_retriever_eval
📁 LangSmith Project: contextual_compression_retriever_eval

⏳ Applying 30-second rate limit delay for Contextual Compression before question 1...
✅ Rate limit delay completed for Contextual Compression - question 1
⏳ Applying 30-second rate limit delay for Contextual Compression before question 2...
✅ Rate limit delay completed for Contextual Compression - question 2
⏳ Applying 30-second rate limit delay for Contextual Compression before question 3...
✅ Rate limit delay completed for Contextual Compression - question 3
⏳ Applying 30-second rate limit delay for Contextual Compression before question 4...
✅ Rate limit delay completed for Contextual Compression - question 4
⏳ Applying 30-second rate limit delay for Contextual Compression before question 5...
✅ Rate l

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[5]: AttributeError('StringIO' object has no attribute 'statements')
Exception raised in Job[41]: AttributeError('StringIO' object has no attribute 'statements')


🔍 DEBUG: Extracting metrics from Ragas result...
🔍 DEBUG: Scores type: <class 'list'>
🔍 DEBUG: Sample scores keys: ['context_recall', 'faithfulness', 'factual_correctness', 'answer_relevancy', 'context_entity_recall', 'noise_sensitivity_relevant']
🔍 DEBUG: Looking for metric: LLMContextRecall
🔍 DEBUG: Looking for key: 'context_recall'
🔍 DEBUG: Found LLMContextRecall = 1.0
🔍 DEBUG: Looking for metric: Faithfulness
🔍 DEBUG: Looking for key: 'faithfulness'
🔍 DEBUG: Found Faithfulness = 0.8
🔍 DEBUG: Looking for metric: FactualCorrectness
🔍 DEBUG: Looking for key: 'factual_correctness'
🔍 DEBUG: Found FactualCorrectness = 0.8
🔍 DEBUG: Looking for metric: ResponseRelevancy
🔍 DEBUG: Looking for key: 'answer_relevancy'
🔍 DEBUG: Found ResponseRelevancy = 0.9999990289520909
🔍 DEBUG: Looking for metric: ContextEntityRecall
🔍 DEBUG: Looking for key: 'context_entity_recall'
🔍 DEBUG: Found ContextEntityRecall = 0.7499999981250001
🔍 DEBUG: Looking for metric: NoiseSensitivity
🔍 DEBUG: Looking for key:

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[41]: AttributeError('StringIO' object has no attribute 'statements')


🔍 DEBUG: Extracting metrics from Ragas result...
🔍 DEBUG: Scores type: <class 'list'>
🔍 DEBUG: Sample scores keys: ['context_recall', 'faithfulness', 'factual_correctness', 'answer_relevancy', 'context_entity_recall', 'noise_sensitivity_relevant']
🔍 DEBUG: Looking for metric: LLMContextRecall
🔍 DEBUG: Looking for key: 'context_recall'
🔍 DEBUG: Found LLMContextRecall = 1.0
🔍 DEBUG: Looking for metric: Faithfulness
🔍 DEBUG: Looking for key: 'faithfulness'
🔍 DEBUG: Found Faithfulness = 0.8571428571428571
🔍 DEBUG: Looking for metric: FactualCorrectness
🔍 DEBUG: Looking for key: 'factual_correctness'
🔍 DEBUG: Found FactualCorrectness = 0.88
🔍 DEBUG: Looking for metric: ResponseRelevancy
🔍 DEBUG: Looking for key: 'answer_relevancy'
🔍 DEBUG: Found ResponseRelevancy = 0.9798272024981207
🔍 DEBUG: Looking for metric: ContextEntityRecall
🔍 DEBUG: Looking for key: 'context_entity_recall'
🔍 DEBUG: Found ContextEntityRecall = 0.7499999981250001
🔍 DEBUG: Looking for metric: NoiseSensitivity
🔍 DEBUG: 

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

🔍 DEBUG: Extracting metrics from Ragas result...
🔍 DEBUG: Scores type: <class 'list'>
🔍 DEBUG: Sample scores keys: ['context_recall', 'faithfulness', 'factual_correctness', 'answer_relevancy', 'context_entity_recall', 'noise_sensitivity_relevant']
🔍 DEBUG: Looking for metric: LLMContextRecall
🔍 DEBUG: Looking for key: 'context_recall'
🔍 DEBUG: Found LLMContextRecall = 1.0
🔍 DEBUG: Looking for metric: Faithfulness
🔍 DEBUG: Looking for key: 'faithfulness'
🔍 DEBUG: Found Faithfulness = 1.0
🔍 DEBUG: Looking for metric: FactualCorrectness
🔍 DEBUG: Looking for key: 'factual_correctness'
🔍 DEBUG: Found FactualCorrectness = 0.78
🔍 DEBUG: Looking for metric: ResponseRelevancy
🔍 DEBUG: Looking for key: 'answer_relevancy'
🔍 DEBUG: Found ResponseRelevancy = 0.0
🔍 DEBUG: Looking for metric: ContextEntityRecall
🔍 DEBUG: Looking for key: 'context_entity_recall'
🔍 DEBUG: Found ContextEntityRecall = 0.249999999375
🔍 DEBUG: Looking for metric: NoiseSensitivity
🔍 DEBUG: Looking for key: 'noise_sensitivit

In [44]:
# Evaluate Ensemble Retrieval (with rate limit handling)
print("\n" + "="*60)
print("6/6: EVALUATING ENSEMBLE RETRIEVAL (with rate limits)")
print("="*60)
result6 = evaluate_retriever_method("Ensemble Retrieval", ensemble_retrieval_pdf_chain, test_set, evaluator_llm, custom_run_config)
all_results.append(result6)

print("\n" + "=" * 80)
print("🎉 ALL EVALUATIONS COMPLETED WITH CORRECTED FUNCTION!")
print("=" * 80)


6/6: EVALUATING ENSEMBLE RETRIEVAL (with rate limits)

Evaluating Ensemble Retrieval...
🗑️  Deleted existing project: ensemble_retrieval_retriever_eval
✅ Created new project: ensemble_retrieval_retriever_eval
📁 LangSmith Project: ensemble_retrieval_retriever_eval

⏳ Applying 30-second rate limit delay for Ensemble Retrieval before question 1...
✅ Rate limit delay completed for Ensemble Retrieval - question 1
⏳ Applying 30-second rate limit delay for Ensemble Retrieval before question 2...
✅ Rate limit delay completed for Ensemble Retrieval - question 2
⏳ Applying 30-second rate limit delay for Ensemble Retrieval before question 3...
✅ Rate limit delay completed for Ensemble Retrieval - question 3
⏳ Applying 30-second rate limit delay for Ensemble Retrieval before question 4...
✅ Rate limit delay completed for Ensemble Retrieval - question 4
⏳ Applying 30-second rate limit delay for Ensemble Retrieval before question 5...
✅ Rate limit delay completed for Ensemble Retrieval - question 5

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[2]: AttributeError('StringIO' object has no attribute 'statements')
Exception raised in Job[5]: AttributeError('StringIO' object has no attribute 'statements')
Exception raised in Job[59]: AttributeError('StringIO' object has no attribute 'statements')
Exception raised in Job[17]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[47]: TimeoutError()
Exception raised in Job[65]: TimeoutError()


🔍 DEBUG: Extracting metrics from Ragas result...
🔍 DEBUG: Scores type: <class 'list'>
🔍 DEBUG: Sample scores keys: ['context_recall', 'faithfulness', 'factual_correctness', 'answer_relevancy', 'context_entity_recall', 'noise_sensitivity_relevant']
🔍 DEBUG: Looking for metric: LLMContextRecall
🔍 DEBUG: Looking for key: 'context_recall'
🔍 DEBUG: Found LLMContextRecall = 1.0
🔍 DEBUG: Looking for metric: Faithfulness
🔍 DEBUG: Looking for key: 'faithfulness'
🔍 DEBUG: Found Faithfulness = 1.0
🔍 DEBUG: Looking for metric: FactualCorrectness
🔍 DEBUG: Looking for key: 'factual_correctness'
🔍 DEBUG: Found FactualCorrectness = nan (converted to None due to NaN)
🔍 DEBUG: Looking for metric: ResponseRelevancy
🔍 DEBUG: Looking for key: 'answer_relevancy'
🔍 DEBUG: Found ResponseRelevancy = 0.9798119412628344
🔍 DEBUG: Looking for metric: ContextEntityRecall
🔍 DEBUG: Looking for key: 'context_entity_recall'
🔍 DEBUG: Found ContextEntityRecall = 0.7499999981250001
🔍 DEBUG: Looking for metric: NoiseSensit

In [48]:
# COMPREHENSIVE RESULTS DISPLAY
print("📊 COMPREHENSIVE EVALUATION RESULTS")
print("=" * 80)

# Create comprehensive results DataFrame
results_df = pd.DataFrame(all_results)

# Display results with all metrics
print("\n📋 FULL RESULTS TABLE:")
print("=" * 100)
print(results_df.to_string(index=False, float_format='%.4f'))

# Save results
results_df.to_csv("golden_dataset/comprehensive_evaluation_results.csv", index=False)
print(f"\n💾 Results saved to golden_dataset/comprehensive_evaluation_results.csv")

# Display Ragas metrics summary
print("\n" + "=" * 80)
print("🎯 RAGAS METRICS SUMMARY")
print("=" * 80)

ragas_columns = ['Method', 'LLMContextRecall', 'Faithfulness', 'FactualCorrectness', 
                'ResponseRelevancy', 'ContextEntityRecall', 'NoiseSensitivity']

if all(col in results_df.columns for col in ragas_columns):
    ragas_summary = results_df[ragas_columns]
    print("\nRagas Metrics Comparison:")
    print("=" * 100)
    print(ragas_summary.to_string(index=False, float_format='%.4f'))
    
    # Calculate efficiency metrics using decimal division
    if 'FactualCorrectness' in results_df.columns and 'Cost_per_Question' in results_df.columns:
        from decimal import Decimal
        results_df['Efficiency_Score'] = results_df.apply(lambda x: Decimal(str(x['FactualCorrectness'])) / Decimal(str(x['Cost_per_Question'] if x['Cost_per_Question'] != 0 else 1)), axis=1)
        results_df['Latency_Efficiency'] = results_df.apply(lambda x: Decimal(str(x['FactualCorrectness'])) / Decimal(str(x['Avg_Latency_ms'] if x['Avg_Latency_ms'] != 0 else 1)), axis=1)
        
        print("\n" + "=" * 80)
        print("⚡ EFFICIENCY RANKINGS (Higher is Better)")
        print("=" * 80)
        efficiency_ranking = results_df.sort_values('Efficiency_Score', ascending=False)[
            ['Method', 'Efficiency_Score', 'Cost_per_Question', 'FactualCorrectness', 'Avg_Latency_ms']
        ]
        print(efficiency_ranking.to_string(index=False, float_format='%.4f'))

# Display LangSmith metrics summary
print("\n" + "=" * 80)
print("💰 LANGSMITH METRICS SUMMARY")
print("=" * 80)

langsmith_columns = ['Method', 'Total_Cost_USD', 'Cost_per_Question', 'Avg_Latency_ms', 
                    'P50_Latency_seconds', 'P99_Latency_seconds', 'Total_Tokens']

# Print the actual columns in results_df to debug
print("\nActual columns in results_df:")
print(results_df.columns.tolist())

# Check which LangSmith columns are missing
missing_columns = [col for col in langsmith_columns if col not in results_df.columns]
print("\nMissing LangSmith columns:")
print(missing_columns)

if all(col in results_df.columns for col in langsmith_columns):
    langsmith_summary = results_df[langsmith_columns]
    print("\nLangSmith Metrics Comparison:")
    print("=" * 100)
    # Format Total_Cost_USD to 6 decimal places, keep others at 4
    print(langsmith_summary.to_string(index=False, formatters={
        'Total_Cost_USD': lambda x: f'{x:.6f}',
        'Cost_per_Question': lambda x: f'{x:.4f}',
        'Avg_Latency_ms': lambda x: f'{x:.4f}',
        'P50_Latency_seconds': lambda x: f'{x:.4f}',
        'P99_Latency_seconds': lambda x: f'{x:.4f}',
        'Total_Tokens': lambda x: f'{x:.4f}'
    }))
else:
    print("\nWarning: Some LangSmith metrics are missing from the results.")

# Performance summary
print("\n" + "=" * 80)
print("🏆 PERFORMANCE SUMMARY")
print("=" * 80)

# Best Ragas metrics
if 'Faithfulness' in results_df.columns:
    best_faithfulness = results_df.loc[results_df['Faithfulness'].idxmax()]
    print(f"🥇 Best Faithfulness: {best_faithfulness['Method']} ({best_faithfulness['Faithfulness']:.4f})")

if 'LLMContextRecall' in results_df.columns:
    best_context_recall = results_df.loc[results_df['LLMContextRecall'].idxmax()]
    print(f"🥇 Best Context Recall: {best_context_recall['Method']} ({best_context_recall['LLMContextRecall']:.4f})")

# Best LangSmith metrics
if 'Cost_per_Question' in results_df.columns:
    best_cost = results_df.loc[results_df['Cost_per_Question'].idxmin()]
    print(f"🥇 Most Cost-Effective: {best_cost['Method']} (${best_cost['Cost_per_Question']:.4f} per question)")

if 'Avg_Latency_ms' in results_df.columns:
    best_latency = results_df.loc[results_df['Avg_Latency_ms'].idxmin()]
    print(f"🥇 Fastest: {best_latency['Method']} ({best_latency['Avg_Latency_ms']:.4f} ms)")

print("\n" + "=" * 80)
print("🎉 COMPREHENSIVE EVALUATION COMPLETED SUCCESSFULLY!")
print("=" * 80)


📊 COMPREHENSIVE EVALUATION RESULTS

📋 FULL RESULTS TABLE:
                   Method  Status  Evaluation_Time_seconds                   Total_Cost_USD  Avg_Latency_ms  Total_Tokens                 Cost_per_Question  LLMContextRecall  Faithfulness  FactualCorrectness  ResponseRelevancy  ContextEntityRecall  NoiseSensitivity  P50_Latency_seconds  P99_Latency_seconds             Prompt_Cost_per_Query         Completion_Cost_per_Query
          Naive Retrieval Success                 528.1506   0.2208592000000000000000000000       2683.0000   227242.0000   0.01840493333333333333333333333            1.0000        0.8182              0.7500             0.9808               0.7500            0.1818               2.6830             416.1196  0.003964666666666666666666666667   0.01444026666666666666666666667
           BM25 Retrieval Success                 312.6001                      0.013953300       3054.5000   114444.0000                       0.001162775            1.0000           NaN   